# Skills Example

This notebook demonstrates how to use a `SKILL.md` file with Wags-LLM.

In [ ]:
import logging
import sys
from collections.abc import Mapping
from pathlib import Path
from typing import Any

from pydantic import BaseModel, ConfigDict

from wags_llm.client.bedrock import BedrockClaudeJsonClient
from wags_llm.registry.base import Registry
from wags_llm.services.structured_task import StructuredTaskRunner
from wags_llm.templates.skill_template import SkillTemplate

logging.basicConfig(
    stream=sys.stdout,
    level=logging.WARNING,
    format="%(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("wags_llm").setLevel(logging.DEBUG)

In [2]:
class VariantCurationSkill(SkillTemplate):
    skill_path = Path("skills/variant_curation_0.1.0.md")

    def build_user_prompt(self, payload: Mapping[str, Any]) -> str:
        variant = payload["variant"]
        disease = payload.get("disease", "cancer")

        return f"""Curate the following variant for {disease}.
Variant:
{variant}

Return concise JSON matching the provided schema:
- clinical_significance: one short sentence
- evidence_level: short label
- supporting_rationale: 2-3 short sentences maximum
"""


class VariantCurationResult(BaseModel):
    model_config = ConfigDict(extra="forbid", use_enum_values=True)  # Required

    clinical_significance: str | None = None
    evidence_level: str | None = None
    supporting_rationale: str | None = None
    error_message: str | None = None

In [3]:
skill = VariantCurationSkill()
skill.name, skill.version

('variant_curation', '0.1.0')

In [ ]:
registry = Registry()
registry.register(skill)

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 350

llm_client = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=MAX_TOKENS,
    temperature=0,
)

task_runner = StructuredTaskRunner(
    client=llm_client,
    registry=registry,
)

wags_llm.skills.registry - DEBUG - Registering skill: name='variant_curation', version='0.1.0'
wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=350, temperature=0.000000
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'


In [ ]:
result = task_runner.execute_skill(
    skill_name="variant_curation",
    skill_version="0.1.0",
    payload={
        "variant": "BRAF V600E",
        "disease": "melanoma",
    },
    response_model=VariantCurationResult,
)
result

wags_llm.skills.base - DEBUG - Loading skill from path: skills/variant_curation_0.1.0.md
wags_llm.skills.base - INFO - Loaded skill from path: skills/variant_curation_0.1.0.md
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 576, 'outputTokens': 185, 'totalTokens': 761, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 4014}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'text': '{"clinical_significance":"BRAF V600E is a pathogenic, therapeutically actionable variant in melanoma associated with sensitivity to BRAF and MEK inhibitors.","evidence_level":"Level A (Validated association)","supporting_rationale":"BRAF V600E is the most common activating mutation in melanoma, present in approximately 50% of cases, and constitutively activates the MAPK signaling pathway. FDA-approved targeted therapies including vemurafenib, dabrafenib (BRAF inhibitors), and trametinib (MEK inhibitor)

VariantCurationResult(clinical_significance='BRAF V600E is a pathogenic, therapeutically actionable variant in melanoma associated with sensitivity to BRAF and MEK inhibitors.', evidence_level='Level A (Validated association)', supporting_rationale='BRAF V600E is the most common activating mutation in melanoma, present in approximately 50% of cases, and constitutively activates the MAPK signaling pathway. FDA-approved targeted therapies including vemurafenib, dabrafenib (BRAF inhibitors), and trametinib (MEK inhibitor) demonstrate significant clinical benefit in BRAF V600E-positive melanoma patients. Multiple phase III clinical trials support improved progression-free and overall survival with BRAF/MEK inhibitor combinations compared to chemotherapy.', error_message=None)

In [6]:
result.model_dump()

{'clinical_significance': 'BRAF V600E is a pathogenic, therapeutically actionable variant in melanoma associated with sensitivity to BRAF and MEK inhibitors.',
 'evidence_level': 'Level A (Validated association)',
 'supporting_rationale': 'BRAF V600E is the most common activating mutation in melanoma, present in approximately 50% of cases, and constitutively activates the MAPK signaling pathway. FDA-approved targeted therapies including vemurafenib, dabrafenib (BRAF inhibitors), and trametinib (MEK inhibitor) demonstrate significant clinical benefit in BRAF V600E-positive melanoma patients. Multiple phase III clinical trials support improved progression-free and overall survival with BRAF/MEK inhibitor combinations compared to chemotherapy.',
 'error_message': None}